### SVM
 - 선을 구성하는 매개변수를 조정해서 데이터들을 구분하는 선을 찾고 이를 기반으로 패턴을 인식
 - 데이터와 패턴들과의 거리(마진)을 최대로 만드는 것이 가장 좋은 결과를 얻음

#### SVM 을 활용한 비만도(BMI) 측정 예측
 - BMI = 몸무게(kg) / 키(m^2)
 - 18.5 <= BMI < 25 일 때 표준 몸무게

#### BMI Data 생성
 - 데이터 획득을 위해 무작위로 2만명의 데이터를 생성
 - 키(cm), 몸무게(Kg), Label(저체중(thin), 정상체중(normal), 비만(fat))

In [1]:
import numpy as np
np.random.seed(42)

In [2]:
def calc_bmi(h, w):
    result = ""
    bmi = w / (h/100)**2
    if bmi < 18.5: result = "thin"
    elif bmi < 25: result = "normal"
    else: result= "fat"
    return result

In [8]:
calc_bmi(170, 60)

'normal'

In [9]:
# 출력파일 준비
fp = open("../Data/bmi.csv", "w", encoding="utf-8")
fp.write("height,weight,label\n")  # 띄어쓰기하면 띄어쓰기 포함돼서 들어감. 파일 만들 때 띄어쓰기가 불규칙하게 들어가면 에러 뜨기 때문에 띄어쓰기 안 주는 게 나음.

# 무작위로 데이터 생성하기
cnt = {"thin":0, "normal":0, "fat":0}

for i in range(20000):
    h = np.random.randint(120, 200)
    w = np.random.randint(35, 80)
    label = calc_bmi(h, w)
    cnt[label] += 1    # label 에 해당하는 result로 1 씩 추가
    fp.write(f"{h},{w},{label}\n")

fp.close()
print("OK", cnt)

OK {'thin': 6338, 'normal': 6045, 'fat': 7617}


#### BMI 공식을 사용하지 않고 BMI 예측

In [10]:
import pandas as pd

In [11]:
tbl = pd.read_csv("../Data/bmi.csv")
tbl.head()

,height,weight,label
0,171,63,normal
1,134,77,fat
2,191,55,thin
3,194,45,thin
4,143,37,thin


In [12]:
tbl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   height  20000 non-null  int64 
 1   weight  20000 non-null  int64 
 2   label   20000 non-null  object
dtypes: int64(2), object(1)
memory usage: 468.9+ KB


In [13]:
tbl.describe()

,height,weight
count,20000.000000,20000.000000
mean,159.455500,57.050800
std,23.091948,12.928626
min,120.000000,35.000000
25%,139.000000,46.000000
50%,160.000000,57.000000
75%,179.000000,68.000000
max,199.000000,79.000000


### 정규화
 : 컬럼별로 최대값을 구해 최대값을 1로 만드는 작업

In [15]:
# height, weight 의 수치가 달라서 정규화 혹은 표준화를 해야 함. 표준화를 써도 되나, 여기선 정규화를 써봄.
w = tbl.weight / tbl.weight.max()
h = tbl.height / tbl.height.max()

lab = tbl.label

In [16]:
# Feature Data - 1차원 데이터는 시리즈 타입
w

0        0.797468
1        0.974684
2        0.696203
3        0.569620
4        0.468354
           ...   
19995    0.810127
19996    0.810127
19997    0.898734
19998    0.569620
19999    0.670886
Name: weight, Length: 20000, dtype: float64

In [17]:
wh = pd.concat([w,h], axis='columns')
wh

,weight,height
0,0.797468,0.859296
1,0.974684,0.673367
2,0.696203,0.959799
3,0.569620,0.974874
4,0.468354,0.718593
...,...,...
19995,0.810127,0.618090
19996,0.810127,0.829146
19997,0.898734,0.663317
19998,0.569620,0.869347


In [18]:
from sklearn.model_selection import train_test_split

In [19]:
train_data, test_data, train_target, test_target = \
    train_test_split(
        wh,
        tbl.label,
        random_state=42,
        stratify=tbl.label
    )

In [26]:
from sklearn.svm import SVC

In [27]:
# 모델 만들기
clf = SVC()

In [31]:
# 학습 시키기
clf.fit(train_data, train_target)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [30]:
# 검증하기
print(clf.score(train_data, train_target))
print(clf.score(test_data, test_target))

0.9966
0.9944
